# Graduate Student Outcomes Report — Generator
Reads the cleaned Parquet file and produces a monthly HTML report for graduate programs.

**Report structure:**
1. Program filter (top-level, controls all sections)
2. Enrolled student section — enrollment trends, GPA, financial support, thesis progression
3. Graduated student section — outcome KPI cards, career outcomes chart, graduate table

**GIT Commits:**
- Commit 5: Enrollment and financial charts
- Commit 6: GPA and thesis progression charts
- Commit 7: Career outcomes section
- Commit 8: HTML report wrapper and auto-save

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from pathlib import Path
from datetime import datetime
import base64
import io
import json
import webbrowser
import tempfile

# NOTE: matplotlib.use("Agg") has been moved into main() so charts
# render inline in the notebook during development.
%matplotlib inline

## Paths

In [2]:
# This notebook lives in /src — go up one level to reach project root
PROJECT_ROOT = Path().resolve().parent

CLEAN_PATH  = PROJECT_ROOT / "data" / "processed" / "grad_students_clean.parquet"
REPORT_DIR  = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Clean data exists:", CLEAN_PATH.exists())

Project root: /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis
Clean data exists: True


In [3]:
# Load cleaned data
df = pd.read_parquet(CLEAN_PATH)
print(f"[LOAD] {len(df):,} rows × {len(df.columns)} columns")

# Build graduates sub-dataset — one row per student who has graduated.
# All KPI cards and career outcome charts pull from this, not from df,
# so we never accidentally use enrollment-term rows for outcome stats.
graduates = df[df["graduated_flag"]].copy()
assert graduates["student_id"].is_unique, "Duplicate student_id in graduates!"
print(f"[LOAD] {len(graduates)} graduates identified")

[LOAD] 550 rows × 41 columns
[LOAD] 42 graduates identified


## Style palette

In [4]:
PALETTE = {
    "primary":   "#CFB87C",  # CU Gold
    "secondary": "#000000",  # CU Black
    "accent":    "#565A5C",  # Dark Gray
    "danger":    "#9D2235",  # Dark Red
    "light":     "#F5F5F5",
    "dark":      "#000000",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F5F5F5",
    "axes.edgecolor":   "#CFB87C",
    "axes.grid":        True,
    "grid.color":       "#D3D3D3",
    "grid.linestyle":   "--",
    "grid.alpha":       0.6,
    "font.family":      "DejaVu Sans",
    "font.size":        10,
    "axes.titlesize":   12,
    "axes.titleweight": "bold",
    "axes.labelcolor":  "#000000",
    "xtick.color":      "#000000",
    "ytick.color":      "#000000",
    "text.color":       "#000000",
})

## Helper: embed figure as base64 PNG

In [5]:
def fig_to_b64(fig) -> str:
    """Save a matplotlib figure to an in-memory PNG and return as base64 string.
    This lets us embed images directly into HTML without saving to disk first."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

def save_figure(fig, filename: str):
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [FIG] Saved {path}")

---
## Section 1 — Enrolled Student Charts
All charts below accept a `programs` list to filter by. When `programs` is empty or None,
they show all programs. This is what the HTML program filter drives.

### Chart 1: Enrollment by Term — Interactive Cards

In [6]:
def chart_enrollment(df: pd.DataFrame) -> str:
    """Interactive enrollment cards — one card per program showing total
    unique student-terms. A term dropdown filters all cards live.
    Returns raw HTML (not a base64 image)."""

    # Build chronologically ordered term list
    tmp = df[["term_label", "term_year", "academic_term"]].drop_duplicates().copy()
    tmp["season"] = tmp["academic_term"].map({"Fall": 1, "Spring": 0})
    terms = tmp.sort_values(["term_year", "season"])["term_label"].tolist()

    # Enrollment counts per term per program
    enroll = (
        df.groupby(["term_label", "program"])["student_id"]
        .nunique()
        .reset_index(name="n_students")
    )
    programs = sorted(df["program"].unique().tolist())

    data_json     = json.dumps(
        enroll.groupby("term_label")
              .apply(lambda g: dict(zip(g["program"], g["n_students"])))
              .to_dict()
    )
    terms_json    = json.dumps(terms)
    programs_json = json.dumps(programs)

    return f"""
    <div id="enrollment-section">
      <div style="margin-bottom:20px;display:flex;align-items:center;gap:16px;flex-wrap:wrap;">
        <label style="font-weight:bold;color:#000;">Filter by Term:</label>
        <select id="term-select" onchange="filterCards()"
                style="padding:8px 14px;border:2px solid #CFB87C;border-radius:6px;
                       font-size:14px;background:white;cursor:pointer;">
          <option value="ALL">All Terms</option>
          {" ".join(f'<option value="{t}">{t}</option>' for t in terms)}
        </select>
        <span id="enroll-total"
              style="background:#CFB87C;color:#000;padding:6px 14px;
                     border-radius:20px;font-size:13px;font-weight:bold;"></span>
      </div>
      <div id="card-grid"
           style="display:grid;grid-template-columns:repeat(auto-fill,minmax(180px,1fr));gap:14px;">
      </div>
    </div>

    <script>
      const ENROLL_DATA = {data_json};
      const ENROLL_TERMS = {terms_json};
      const ENROLL_PROGRAMS = {programs_json};

      function getEnrollCount(term, program) {{
        if (term === "ALL")
          return ENROLL_TERMS.reduce((a,t) => a + ((ENROLL_DATA[t]||{{}})[program]||0), 0);
        return (ENROLL_DATA[term]||{{}})[program] || 0;
      }}

      function filterCards() {{
        const term = document.getElementById("term-select").value;
        const grid = document.getElementById("card-grid");
        const selectedPrograms = getSelectedPrograms();
        grid.innerHTML = "";
        let total = 0;
        ENROLL_PROGRAMS
          .filter(p => selectedPrograms.length === 0 || selectedPrograms.includes(p))
          .forEach(p => {{
            const n = getEnrollCount(term, p);
            total += n;
            const card = document.createElement("div");
            card.style.cssText = `background:white;border-radius:10px;padding:16px 18px;
              box-shadow:0 2px 8px rgba(0,0,0,0.08);border-left:5px solid #CFB87C;
              opacity:${{n===0?0.3:1}};`;
            card.innerHTML = `<div style="font-size:1.8rem;font-weight:700;color:#CFB87C;">${{n}}</div>
              <div style="font-size:0.78rem;color:#555;margin-top:4px;line-height:1.4;">${{p}}</div>`;
            grid.appendChild(card);
          }});
        document.getElementById("enroll-total").textContent = "Total: " + total + " student-terms";
      }}
      window.filterCards = filterCards;
    </script>
    """

### Chart 2: Financial Support & Equity Indicators

In [7]:
def chart_financial_support(df: pd.DataFrame) -> str:
    """Two panels: Pell/First-Gen rates by program (grouped bar) and
    total financial support distribution (box plot). Returns base64 PNG."""

    summary = (
        df.drop_duplicates("student_id")
        .groupby("program")[["pell_eligible", "first_gen"]]
        .mean()
        .mul(100).round(1)
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    # Panel A: Pell & First-Gen rates
    x = np.arange(len(summary))
    w = 0.35
    ax1.bar(x - w/2, summary["pell_eligible"], w, label="Pell-Eligible",
            color=PALETTE["primary"], alpha=0.85)
    ax1.bar(x + w/2, summary["first_gen"],     w, label="First-Generation",
            color=PALETTE["secondary"], alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(summary.index, rotation=35, ha="right", fontsize=7)
    ax1.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax1.set_title("Pell-Eligible & First-Gen Rate by Program")
    ax1.set_ylabel("% of Students")
    ax1.legend()

    # Panel B: Financial support distribution
    programs = df["program"].unique()
    data_by_prog = [
        df[df["program"] == p]["total_financial_support"].dropna().values
        for p in programs
    ]
    bp = ax2.boxplot(data_by_prog, patch_artist=True,
                     medianprops={"color": "black", "linewidth": 2})
    for patch in bp["boxes"]:
        patch.set_facecolor(PALETTE["primary"])
        patch.set_alpha(0.65)
    ax2.set_xticks(range(1, len(programs)+1))
    ax2.set_xticklabels(programs, rotation=35, ha="right", fontsize=7)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))
    ax2.set_title("Total Financial Support Per Term (Grant + Stipend)")
    ax2.set_ylabel("$")

    fig.tight_layout()
    b64 = fig_to_b64(fig)
    save_figure(fig, "02_financial_support.png")
    return b64

### Chart 3: GPA Trends & Academic Risk

In [8]:
def chart_gpa(df: pd.DataFrame) -> str:
    """Line chart of mean GPA per term, one line per program.
    Red dashed line marks the 3.0 academic probation threshold.
    Only shows the top 3 programs by enrollment for readability;
    when filtered, shows all programs in the selection."""

    term_gpa = df.groupby(["term_label", "program", "term_year", "academic_term"])["gpa"] \
                 .mean().reset_index()
    term_gpa["season"] = term_gpa["academic_term"].map({"Fall": 1, "Spring": 0})
    term_gpa = term_gpa.sort_values(["term_year", "season"])

    # Limit to top 3 programs by row count for readability in the unfiltered view
    top_programs = df["program"].value_counts().head(3).index.tolist()
    colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]]

    fig, ax = plt.subplots(figsize=(11, 4))
    for prog, color in zip(top_programs, colors):
        sub = term_gpa[term_gpa["program"] == prog].drop_duplicates("term_label")
        ax.plot(sub["term_label"], sub["gpa"], marker="o", label=prog, color=color)

    ax.axhline(3.0, color=PALETTE["danger"], linestyle="--", linewidth=1.5,
               label="Probation threshold (3.0)")
    ax.set_title("Mean GPA Trend by Program")
    ax.set_xlabel("Academic Term")
    ax.set_ylabel("Mean GPA")
    ax.set_ylim(2.0, 4.2)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    fig.tight_layout()
    b64 = fig_to_b64(fig)
    save_figure(fig, "03_gpa_trend.png")
    return b64

### Chart 4: PhD Thesis Stage Progression

In [9]:
def chart_thesis(df: pd.DataFrame) -> str:
    """Heatmap of PhD thesis stage counts per term.
    FIX: degree_program_type is stored as 'PHD' (all caps) in the parquet,
    so the filter must use 'PHD', not 'PhD'."""

    # NOTE: the original code used degree_program_type == "PhD" which produced
    # an empty heatmap because the column stores "PHD" (all caps).
    # Fixed here by uppercasing the comparison.
    phd = df[df["degree_program_type"].str.upper() == "PHD"].copy()

    if phd.empty:
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.text(0.5, 0.5, "No PhD students in selected filter",
                ha="center", va="center", transform=ax.transAxes)
        b64 = fig_to_b64(fig)
        plt.close(fig)
        return b64

    stage_labels = {0: "N/A", 1: "Proposal", 2: "Approved", 3: "Writing", 4: "Defended"}
    phd["stage_label"] = phd["thesis_stage_num"].map(stage_labels)

    pivot = (
        phd.groupby(["term_label", "stage_label"])["student_id"]
        .nunique()
        .unstack(fill_value=0)
        .reindex(columns=["Proposal", "Approved", "Writing", "Defended"], fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.imshow(pivot.T.values, cmap="YlOrBr", aspect="auto")
    ax.set_xticks(range(len(pivot.index)))
    ax.set_xticklabels(pivot.index, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.columns)))
    ax.set_yticklabels(pivot.columns)
    ax.set_title("PhD Thesis Stage Progression (# Students per Cell)")
    plt.colorbar(im, ax=ax, label="# Students")

    for i in range(len(pivot.columns)):
        for j in range(len(pivot.index)):
            val = pivot.T.values[i, j]
            if val > 0:
                ax.text(j, i, str(val), ha="center", va="center",
                        fontsize=8, color="black" if val < 4 else "white")
    fig.tight_layout()
    b64 = fig_to_b64(fig)
    save_figure(fig, "04_thesis_progression.png")
    return b64

---
## Section 2 — Graduated Student Charts & KPIs
All charts and cards below pull from the `graduates` sub-dataset (one row per
graduated student), never from the full longitudinal `df`. This ensures KPIs
like employment rate and median salary are computed on actual graduates only,
not on all enrollment-term rows (which would show 0% employment and $nan salary).

### Graduated Student KPI Cards (filterable by program)

In [10]:
def build_grad_kpi_cards(graduates: pd.DataFrame) -> str:
    """Outcome KPI cards computed from the graduates sub-dataset.
    These update via JavaScript when the program filter changes.

    FIX: Previous version computed these from df.drop_duplicates('student_id'),
    which kept each student's FIRST enrollment row — before graduation —
    resulting in 0% employment rate and $nan median salary.
    Now correctly uses the graduates sub-dataset."""

    programs = sorted(graduates["program"].unique().tolist())

    # Build per-program stats for JS filtering
    stats = {}
    for prog in programs + ["ALL"]:
        subset = graduates if prog == "ALL" else graduates[graduates["program"] == prog]
        n = len(subset)
        emp_rate = f"{subset['employed_flag'].mean() * 100:.0f}%" if n > 0 else "—"
        med_sal  = f"${subset['salary_offer'].median():,.0f}" if subset['salary_offer'].notna().any() else "—"
        mean_ttd = f"{subset['time_to_degree'].mean():.1f} yrs" if subset['time_to_degree'].notna().any() else "—"
        stats[prog] = {
            "n":        n,
            "emp_rate": emp_rate,
            "med_sal":  med_sal,
            "mean_ttd": mean_ttd,
        }

    stats_json = json.dumps(stats)

    return f"""
    <div id="kpi-row" style="display:grid;grid-template-columns:repeat(auto-fill,minmax(170px,1fr));
                             gap:14px;margin-bottom:8px;"></div>
    <script>
      const GRAD_STATS = {stats_json};

      function updateKpiCards(program) {{
        const s = GRAD_STATS[program] || GRAD_STATS["ALL"];
        const row = document.getElementById("kpi-row");
        row.innerHTML = [
          [s.n,        "Graduates"],
          [s.emp_rate, "Employment Rate"],
          [s.med_sal,  "Median Starting Salary"],
          [s.mean_ttd, "Avg. Time to Degree"],
        ].map(([val, lbl]) => `
          <div style="background:white;border-radius:10px;padding:18px 20px;
                      box-shadow:0 2px 6px rgba(0,0,0,0.07);border-left:5px solid #CFB87C;">
            <div style="font-size:1.6rem;font-weight:700;color:#CFB87C;">${{val}}</div>
            <div style="font-size:0.75rem;color:#666;text-transform:uppercase;
                        letter-spacing:.05em;margin-top:5px;">${{lbl}}</div>
          </div>`).join("");
      }}
      window.updateKpiCards = updateKpiCards;
      updateKpiCards("ALL");
    </script>
    """

### Chart 5: Career Outcomes Dashboard

In [11]:
def chart_career(graduates: pd.DataFrame) -> str:
    """Three panels: employer type pie, mean salary by employer type,
    and months-to-employment histogram.

    FIX: now accepts and uses the `graduates` sub-dataset directly,
    not the full df. This ensures charts only reflect actual graduates."""

    employed = graduates[graduates["employment_outcome"].str.title() == "Employed"].copy()

    if employed.empty:
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.text(0.5, 0.5, "No employed graduates in selected filter",
                ha="center", va="center", transform=ax.transAxes)
        b64 = fig_to_b64(fig)
        plt.close(fig)
        return b64

    fig = plt.figure(figsize=(14, 4))
    gs  = GridSpec(1, 3, figure=fig, wspace=0.38)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax3 = fig.add_subplot(gs[2])

    # Panel A: Employer type pie
    emp_counts = employed["employer_type"].value_counts()
    pie_colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"],
                  PALETTE["danger"], "#888888", "#CCCCCC"]
    ax1.pie(emp_counts, labels=emp_counts.index, autopct="%1.0f%%",
            colors=pie_colors[:len(emp_counts)], startangle=140,
            textprops={"fontsize": 8})
    ax1.set_title("Employer Type Distribution")

    # Panel B: Mean salary by employer type
    sal = (employed.groupby("employer_type")["salary_offer"]
                   .agg(["mean", "std"]).sort_values("mean", ascending=True))
    ax2.barh(sal.index, sal["mean"], xerr=sal["std"].fillna(0),
             color=PALETTE["primary"], alpha=0.85)
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x/1000:.0f}K"))
    ax2.set_title("Mean Salary by Employer Type")
    ax2.set_xlabel("Annual Salary")

    # Panel C: Months to employment
    months = employed["months_to_employment"].dropna()
    ax3.hist(months, bins=8, color=PALETTE["accent"], edgecolor="white", alpha=0.85)
    ax3.set_title("Months to Employment After Graduation")
    ax3.set_xlabel("Months")
    ax3.set_ylabel("# Graduates")
    if len(months) > 0:
        ax3.axvline(months.mean(), color=PALETTE["danger"], linestyle="--",
                    label=f"Mean: {months.mean():.1f} mo")
        ax3.legend()

    fig.tight_layout()
    b64 = fig_to_b64(fig)
    save_figure(fig, "05_career_outcomes.png")
    return b64

### Graduated Student Table

In [12]:
def build_grad_table(graduates: pd.DataFrame) -> str:
    """HTML table of graduated students with key outcome columns.

    FIX: Previous version used df.drop_duplicates('student_id') and filtered
    for thesis_status == 'Defended', which returned zero rows because
    drop_duplicates kept each student's *first* enrollment row (Proposal stage),
    not their graduating row. Now uses the `graduates` sub-dataset directly."""

    cols = ["student_id", "last_name", "first_name", "program",
            "time_to_degree", "employment_outcome", "employer_type",
            "salary_offer", "months_to_employment"]

    table = graduates[cols].sort_values(["program", "last_name"]).copy()

    rows = ""
    for _, r in table.iterrows():
        sal  = f"${r['salary_offer']:,.0f}"  if pd.notna(r['salary_offer'])        else "—"
        ttd  = f"{r['time_to_degree']:.1f}"  if pd.notna(r['time_to_degree'])      else "—"
        mo   = f"{r['months_to_employment']:.0f} mo" if pd.notna(r['months_to_employment']) else "—"
        prog_attr = r['program'].replace('"', '')
        rows += f"""
        <tr data-program="{prog_attr}">
          <td>{r['student_id']}</td>
          <td>{r['last_name']}, {r['first_name']}</td>
          <td>{r['program']}</td>
          <td>{ttd}</td>
          <td>{r['employment_outcome'] if pd.notna(r['employment_outcome']) else '—'}</td>
          <td>{r['employer_type'] if pd.notna(r['employer_type']) else '—'}</td>
          <td>{sal}</td>
          <td>{mo}</td>
        </tr>"""

    return f"""
    <div id="grad-table-count" style="font-size:0.85rem;color:#666;margin-bottom:10px;"></div>
    <div style="overflow-x:auto;">
    <table id="grad-table" style="width:100%;border-collapse:collapse;font-size:0.86rem;">
      <thead>
        <tr style="background:#CFB87C;color:#000;">
          <th style="padding:10px 12px;text-align:left;">ID</th>
          <th style="padding:10px 12px;text-align:left;">Name</th>
          <th style="padding:10px 12px;text-align:left;">Program</th>
          <th style="padding:10px 12px;text-align:left;">Time to Degree</th>
          <th style="padding:10px 12px;text-align:left;">Outcome</th>
          <th style="padding:10px 12px;text-align:left;">Employer Type</th>
          <th style="padding:10px 12px;text-align:left;">Starting Salary</th>
          <th style="padding:10px 12px;text-align:left;">Months to Employment</th>
        </tr>
      </thead>
      <tbody id="grad-tbody">{rows}</tbody>
    </table>
    </div>
    <script>
      function filterGradTable(program) {{
        const rows = document.querySelectorAll("#grad-tbody tr");
        let visible = 0;
        rows.forEach(row => {{
          const match = program === "ALL" || row.dataset.program === program;
          row.style.display = match ? "" : "none";
          if (match) visible++;
        }});
        document.getElementById("grad-table-count").textContent =
          `Showing ${{visible}} graduate${{visible !== 1 ? "s" : ""}}`;
      }}
      window.filterGradTable = filterGradTable;
      filterGradTable("ALL");
    </script>
    """

## HTML Report Builder

In [13]:
def build_html_report(df: pd.DataFrame, graduates: pd.DataFrame,
                      charts: dict, run_date: str) -> str:
    """Assemble the full HTML report.

    Structure:
      - Global program filter at the top (controls all sections via JS)
      - Section 1: Enrolled students (enrollment cards, financial, GPA, thesis)
      - Section 2: Graduated students (KPI cards, career outcomes, graduate table)

    The program filter drives:
      - Enrollment cards (via filterCards())
      - KPI cards (via updateKpiCards())
      - Graduate table (via filterGradTable())
    Matplotlib charts are static images — for per-program filtering on those,
    re-run main() after filtering df and graduates in Python.
    """

    n_students  = df["student_id"].nunique()
    programs    = sorted(df["program"].unique().tolist())

    def img_tag(b64, alt):
        return (f'<img src="data:image/png;base64,{b64}" alt="{alt}" '
                f'style="width:100%;max-width:900px;border-radius:6px;'
                f'box-shadow:0 2px 8px rgba(0,0,0,0.1);">')

    program_options = '\n'.join(
        f'<option value="{p}">{p}</option>' for p in programs
    )

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Graduate Student Outcomes Report – {run_date}</title>
<style>
  * {{ box-sizing:border-box; margin:0; padding:0; }}
  body {{ font-family:'Segoe UI',Arial,sans-serif; background:#F5F5F5; color:#1a1a1a; }}
  header {{ background:#000; color:#CFB87C; padding:24px 40px; }}
  header h1 {{ font-size:1.7rem; font-weight:700; }}
  header p  {{ font-size:0.85rem; margin-top:4px; opacity:0.75; color:white; }}
  main {{ max-width:1100px; margin:28px auto; padding:0 24px; }}

  /* Global filter bar */
  .filter-bar {{ background:white; border-radius:10px; padding:16px 24px;
                 margin-bottom:28px; box-shadow:0 2px 6px rgba(0,0,0,0.07);
                 display:flex; align-items:center; gap:16px; flex-wrap:wrap; }}
  .filter-bar label {{ font-weight:700; font-size:0.95rem; }}
  .filter-bar select {{ padding:8px 16px; border:2px solid #CFB87C; border-radius:6px;
                        font-size:14px; background:white; cursor:pointer; min-width:220px; }}
  .filter-badge {{ background:#CFB87C; color:#000; padding:6px 14px;
                   border-radius:20px; font-size:12px; font-weight:700; }}

  /* Section containers */
  .section-header {{ font-size:1rem; font-weight:700; color:#CFB87C;
                     text-transform:uppercase; letter-spacing:.08em;
                     border-bottom:3px solid #CFB87C; padding-bottom:8px;
                     margin:32px 0 18px; }}
  .card {{ background:white; border-radius:10px; padding:22px 26px;
            margin-bottom:22px; box-shadow:0 2px 6px rgba(0,0,0,0.07); }}
  .card h2 {{ font-size:1rem; color:#000; margin-bottom:16px;
               border-bottom:1px solid #E0E0E0; padding-bottom:10px; }}

  /* Table */
  table {{ width:100%; border-collapse:collapse; font-size:0.86rem; }}
  th {{ background:#CFB87C; color:#000; padding:10px 12px; text-align:left;
        font-weight:700; }}
  td {{ padding:9px 12px; border-bottom:1px solid #EEEEEE; }}
  tr:hover td {{ background:#FBF8F0; }}
  footer {{ text-align:center; padding:24px; font-size:0.8rem; color:#999; }}
</style>
</head>
<body>
<header>
  <h1>Graduate Student Outcomes Report</h1>
  <p>Generated: {run_date} &nbsp;|&nbsp; Total students tracked: {n_students}</p>
</header>
<main>

  <!-- ── GLOBAL PROGRAM FILTER ── -->
  <div class="filter-bar">
    <label>Filter by Program:</label>
    <select id="program-select" onchange="applyProgramFilter()">
      <option value="ALL">All Programs</option>
      {program_options}
    </select>
    <span class="filter-badge" id="filter-label">Showing all programs</span>
  </div>

  <!-- ══════════════════════════════════════════════════
       SECTION 1 — ENROLLED STUDENTS
  ══════════════════════════════════════════════════ -->
  <div class="section-header">Section 1 — Enrolled Students</div>

  <div class="card">
    <h2>Enrollment by Term & Program</h2>
    {charts['enrollment']}
  </div>

  <div class="card">
    <h2>Financial Support & Equity Indicators</h2>
    {img_tag(charts['financial'], 'Financial Support')}
  </div>

  <div class="card">
    <h2>GPA Trends & Academic Risk</h2>
    {img_tag(charts['gpa'], 'GPA Trends')}
  </div>

  <div class="card">
    <h2>PhD Thesis Stage Progression</h2>
    {img_tag(charts['thesis'], 'Thesis Progression')}
  </div>

  <!-- ══════════════════════════════════════════════════
       SECTION 2 — GRADUATED STUDENTS
  ══════════════════════════════════════════════════ -->
  <div class="section-header">Section 2 — Graduated Students</div>

  <div class="card">
    <h2>Outcome Summary</h2>
    {charts['kpi_cards']}
  </div>

  <div class="card">
    <h2>Post-Graduation Career Outcomes</h2>
    {img_tag(charts['career'], 'Career Outcomes')}
  </div>

  <div class="card">
    <h2>Graduated Students</h2>
    {charts['grad_table']}
  </div>

</main>
<footer>Graduate Student Research Office &nbsp;|&nbsp; Auto-generated by generate_report.ipynb</footer>

<script>
  function applyProgramFilter() {{
    const prog = document.getElementById("program-select").value;
    const label = document.getElementById("filter-label");
    label.textContent = prog === "ALL" ? "Showing all programs" : prog;

    // Drive enrollment cards term filter
    if (typeof filterCards === "function") filterCards();

    // Drive graduated student KPI cards
    if (typeof updateKpiCards === "function") updateKpiCards(prog);

    // Drive graduated student table
    if (typeof filterGradTable === "function") filterGradTable(prog);
  }}

  // Override filterCards to respect the program filter as well
  function getSelectedPrograms() {{
    const prog = document.getElementById("program-select").value;
    return prog === "ALL" ? [] : [prog];
  }}

  // Initialise on load
  applyProgramFilter();
</script>

</body>
</html>"""

## Live Browser Preview

In [14]:
def preview_in_browser(df, graduates):
    """Build the full report and open it in your browser.
    Re-run this cell + refresh the tab to see updates."""
    run_date = datetime.now().strftime("%B %d, %Y")

    charts = {
        "enrollment": chart_enrollment(df),
        "financial":  chart_financial_support(df),
        "gpa":        chart_gpa(df),
        "thesis":     chart_thesis(df),
        "kpi_cards":  build_grad_kpi_cards(graduates),
        "career":     chart_career(graduates),
        "grad_table": build_grad_table(graduates),
    }

    html = build_html_report(df, graduates, charts, run_date)

    tmp_path = Path(tempfile.gettempdir()) / "grad_report_preview.html"
    tmp_path.write_text(html, encoding="utf-8")
    webbrowser.open(f"file://{tmp_path}")
    print(f"[PREVIEW] Opened → {tmp_path}")
    print(f"[PREVIEW] Re-run this cell and refresh your browser tab to update.")


preview_in_browser(df, graduates)

  [FIG] Saved /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/reports/figures/02_financial_support.png


/var/folders/0m/lvyh88t114j_252mgdtb5t6w0000gn/T/ipykernel_61161/2721397115.py:27: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")


  [FIG] Saved /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/reports/figures/03_gpa_trend.png
  [FIG] Saved /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/reports/figures/04_thesis_progression.png


/var/folders/0m/lvyh88t114j_252mgdtb5t6w0000gn/T/ipykernel_61161/246117169.py:53: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  [FIG] Saved /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/reports/figures/05_career_outcomes.png
[PREVIEW] Opened → /var/folders/0m/lvyh88t114j_252mgdtb5t6w0000gn/T/grad_report_preview.html
[PREVIEW] Re-run this cell and refresh your browser tab to update.


## Main — Save Final Report

In [15]:
def main():
    """Full pipeline: load data, generate all charts, save HTML report."""
    matplotlib.use("Agg")   # headless mode for automated saves — only active in main()

    print("=" * 55)
    print("  Graduate Student Monthly Report Generator")
    print("=" * 55)

    df_main       = pd.read_parquet(CLEAN_PATH)
    grads_main    = df_main[df_main["graduated_flag"]].copy()
    run_date      = datetime.now().strftime("%B %d, %Y")
    slug          = datetime.now().strftime("%Y_%m")

    print("[CHARTS] Generating figures...")
    charts = {
        "enrollment": chart_enrollment(df_main),
        "financial":  chart_financial_support(df_main),
        "gpa":        chart_gpa(df_main),
        "thesis":     chart_thesis(df_main),
        "kpi_cards":  build_grad_kpi_cards(grads_main),
        "career":     chart_career(grads_main),
        "grad_table": build_grad_table(grads_main),
    }

    print("[REPORT] Building HTML report...")
    html = build_html_report(df_main, grads_main, charts, run_date)

    report_path = REPORT_DIR / f"grad_outcomes_report_{slug}.html"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"[DONE] Report saved → {report_path}")


if __name__ == "__main__":
    main()

  Graduate Student Monthly Report Generator
[CHARTS] Generating figures...
  [FIG] Saved /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/reports/figures/02_financial_support.png


/var/folders/0m/lvyh88t114j_252mgdtb5t6w0000gn/T/ipykernel_61161/2721397115.py:27: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")


  [FIG] Saved /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/reports/figures/03_gpa_trend.png
  [FIG] Saved /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/reports/figures/04_thesis_progression.png


/var/folders/0m/lvyh88t114j_252mgdtb5t6w0000gn/T/ipykernel_61161/246117169.py:53: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  [FIG] Saved /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/reports/figures/05_career_outcomes.png
[REPORT] Building HTML report...
[DONE] Report saved → /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/reports/grad_outcomes_report_2026_06.html
